# **폐기물 데이터 통합 및 전처리**

| 데이터 프레임명 (변수명) | 데이터 정보 |
| :--- | :--- |
| **`df_수거`** | 쓰레기_수거현황(2020년이후) |
| **`df_1인당`** | 주민1인당_생활계폐기물배출량 |
| **`df_생활계`** | 생활계폐기물_발생량및처리현황(2020년이후) |
| **`df_사업장`** | 사업장배출시설계폐기물_발생량및처리현황(2020년이후) |
| **`df_건설`** | 건설폐기물_발생량및처리현황(2020년이후) |
| **`df_지정`** | 지정폐기물_발생량및처리현황(2020년이후) |

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
plt.rc('font',family='Malgun Gothic' )
!pip install koreanize-matplotlib
from IPython.display import display
import koreanize_matplotlib
import seaborn as sns

In [2]:
# 파일 경로 설정
PATH = {
    '수거':   '쓰레기_수거현황(2020년이후).csv',
    '1인당':  '주민1인당_생활계폐기물배출량.csv',
    '생활계': '생활계폐기물_발생량및처리현황(2020년이후).csv',
    '사업장': '사업장배출시설계폐기물_발생량및처리현황(2020년이후).csv',
    '건설':   '건설폐기물_발생량및처리현황(2020년이후).csv',
    '지정':   '지정폐기물_발생량및처리현황(2020년이후).csv',
}

In [3]:
# 공통 함수: 헤더 설명 row 제거, 불필요한 행 제거, '-' → '0' 치환
def load_csv(path, col1_keep, col2_drop):
    """
    path      : CSV 파일 경로
    col1_keep : 1번 컬럼에서 남길 값 (예: '합계', '계', '서울시')
    col2_drop : 2번 컬럼에서 제거할 값 (예: ['소계', '처리비율'])
    """
    df = pd.read_csv(path)
    df = df.iloc[1:]                              # 첫 번째 row(헤더 설명) 제거
    df = df[df.iloc[:, 0] == col1_keep]           # 전체 합계 행만 유지
    df = df[~df.iloc[:, 1].isin(col2_drop)]       # 서울 전체·비율 행 제거
    df = df.replace('-', '0')                     # '-' → '0' 으로 치환
    return df

In [4]:
# 1. 쓰레기 수거 현황
df_수거 = load_csv(PATH['수거'], col1_keep='합계', col2_drop=['소계'])

df_수거 = df_수거.rename(columns={
    '자치구별(2)':          '자치구',
    '시점':                '연도',
    '배출량(C)':            '쓰레기_배출량',
    '처리방법 (톤/년).1':   '쓰레기_매립량',
    '처리방법 (톤/년).2':   '쓰레기_소각량',
    '처리방법 (톤/년).3':   '쓰레기_재활용량',
    '처리방법 (톤/년).4':   '쓰레기_기타',
})

df_수거 = df_수거[['자치구', '연도', '쓰레기_배출량',
                   '쓰레기_재활용량', '쓰레기_소각량', '쓰레기_매립량', '쓰레기_기타']]

In [5]:
# 2. 주민 1인당 생활계 폐기물 배출량 (2020년 이후 필터링)
df_1인당 = pd.read_csv(PATH['1인당'])
df_1인당 = df_1인당[df_1인당['자치구별(1)'] != '계']   # 서울 전체 제거
df_1인당 = df_1인당[df_1인당['시점'] >= 2020]          # 2020년 이후만
df_1인당 = df_1인당.replace('-', '0')

df_1인당 = df_1인당.rename(columns={
    '자치구별(1)':                              '자치구',
    '시점':                                    '연도',
    '주민 1인당 생활폐기물(쓰레기) 배출량 (㎏/인 일)': '생활폐기물_1인당배출량',
    '주민수 (명)':                              '주민수',
})

df_1인당 = df_1인당[['자치구', '연도', '생활폐기물_1인당배출량', '주민수']]

In [ ]:
# 3. 생활계 폐기물 발생량 및 처리현황
df_생활계 = load_csv(PATH['생활계'], col1_keep='계', col2_drop=['소계', '처리비율'])

df_생활계 = df_생활계.rename(columns={
    '구분별(2)':  '자치구',
    '시점':      '연도',
    '발생량':    '생활폐기물_발생량',
    '재활용':    '생활폐기물_재활용량',
    '재활용.1':  '생활폐기물_일반재활용량',
    '재활용.2':  '생활폐기물_음식물재활용량',
    '소각':      '생활폐기물_소각량',
    '매립':      '생활폐기물_매립량',
    '기타':      '생활폐기물_기타',
})

df_생활계 = df_생활계[['자치구', '연도', '생활폐기물_발생량', '생활폐기물_재활용량',
                       '생활폐기물_일반재활용량', '생활폐기물_음식물재활용량',
                       '생활폐기물_소각량', '생활폐기물_매립량', '생활폐기물_기타']]

In [7]:
# 4. 사업장배출시설계 폐기물 발생량 및 처리현황
df_사업장 = load_csv(PATH['사업장'], col1_keep='합계', col2_drop=['소계'])

df_사업장 = df_사업장.rename(columns={
    '자치구별(2)':  '자치구',
    '시점':        '연도',
    '발생량':      '사업장폐기물_발생량',
    '처리방법':    '사업장폐기물_재활용량',
    '처리방법.1':  '사업장폐기물_소각량',
    '처리방법.2':  '사업장폐기물_매립량',
    '처리방법.3':  '사업장폐기물_기타',
})

df_사업장 = df_사업장[['자치구', '연도', '사업장폐기물_발생량', '사업장폐기물_재활용량',
                       '사업장폐기물_소각량', '사업장폐기물_매립량', '사업장폐기물_기타']]

In [8]:
# 5. 건설폐기물 발생량 및 처리현황
df_건설 = load_csv(PATH['건설'], col1_keep='합계', col2_drop=['소계'])

df_건설 = df_건설.rename(columns={
    '자치구별(2)':  '자치구',
    '시점':        '연도',
    '발생량':      '건설폐기물_발생량',
    '처리방법':    '건설폐기물_재활용량',
    '처리방법.1':  '건설폐기물_소각량',
    '처리방법.2':  '건설폐기물_매립량',
    '처리방법.3':  '건설폐기물_기타',
})

df_건설 = df_건설[['자치구', '연도', '건설폐기물_발생량', '건설폐기물_재활용량',
                   '건설폐기물_소각량', '건설폐기물_매립량', '건설폐기물_기타']]

In [9]:
# 6. 지정폐기물 발생량 및 처리현황
df_지정 = load_csv(PATH['지정'], col1_keep='서울시', col2_drop=['소계'])

df_지정 = df_지정.rename(columns={
    '자치구별(2)':  '자치구',
    '시점':        '연도',
    '발생량':      '지정폐기물_발생량',
    '처리방법':    '지정폐기물_재활용량',
    '처리방법.1':  '지정폐기물_소각량',
    '처리방법.2':  '지정폐기물_매립량',
    '처리방법.3':  '지정폐기물_기타',
})

df_지정 = df_지정[['자치구', '연도', '지정폐기물_발생량', '지정폐기물_재활용량',
                   '지정폐기물_소각량', '지정폐기물_매립량', '지정폐기물_기타']]

In [10]:
# 7. 자치구·연도 타입 통일 후 6개 데이터셋 병합
dfs = [df_수거, df_1인당, df_생활계, df_사업장, df_건설, df_지정]

for df in dfs:
    df['자치구'] = df['자치구'].astype(str).str.strip()
    df['연도']   = df['연도'].astype(int)

# 순서대로 outer join (자치구 + 연도 기준)
result = df_수거
for df in [df_1인당, df_생활계, df_사업장, df_건설, df_지정]:
    result = result.merge(df, on=['자치구', '연도'], how='outer')

result = result.sort_values(['자치구', '연도']).reset_index(drop=True)

In [11]:
# 8. 숫자형 변환 (문자로 읽힌 컬럼들을 숫자로)
for col in result.columns:
    if col not in ['자치구']:
        result[col] = pd.to_numeric(result[col], errors='coerce')

In [12]:
# 9. 저장
result.to_csv('서울시_자치구별_폐기물_통합데이터.csv', index=False, encoding='utf-8-sig')

print(f"완료! shape: {result.shape}")
print(f"결측치: {result.isnull().sum().sum()}개")
print(f"자치구 수: {result['자치구'].nunique()}개")
print(f"연도 범위: {sorted(result['연도'].unique().tolist())}")

완료! shape: (125, 31)
결측치: 0개
자치구 수: 25개
연도 범위: [2020, 2021, 2022, 2023, 2024]
